# IMPORTS

In [13]:
import glob
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks
import numpy as np
from sklearn import tree
from scipy.ndimage import gaussian_filter1d
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier,export_graphviz
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, LeaveOneOut
from scipy.interpolate import interp1d
import neurokit2 as nk

# FILTERS

In [14]:
def butter_lowpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    y = filtfilt(b, a, data)
    return y

def butter_highpass_filter(data, cutoff, fs, order=2):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='highpass', analog=False)
    y = filtfilt(b, a, data)
    return y

def bandpass_filter(data, lowcut, highcut, fs, order=2):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return filtfilt(b, a, data)

# Upsampling signals

Upsample the signal to new sampling rate fs_new.

time_orig: original time vector in seconds (1D array or Series)

signal_orig: original signal values (1D array or Series)

fs_new: target sampling rate in Hz

## Returns:

time_new: new time vector at fs_new

signal_new: interpolated signal at time_new

In [15]:
def upsample_signal(time_orig, signal_orig, fs_new=15):

    duration = time_orig[-1] - time_orig[0]
    n_samples_new = int(duration * fs_new) + 1

    time_new = np.linspace(time_orig[0], time_orig[-1], n_samples_new)

    # Interpolator
    interpolator = interp1d(time_orig, signal_orig, kind='linear', fill_value="extrapolate")

    signal_new = interpolator(time_new)

    return time_new, signal_new

# EA Processing

In [16]:
def ea_detection(csv_file_path, fs=15):
    df = pd.read_csv(csv_file_path)
    df['time[s]'] = (df['LocalTimestamp'] - df['LocalTimestamp'].iloc[0])
    df = df.loc[(df['time[s]'] > 120) & (df['time[s]'] < (df['time[s]'].iloc[-1]) - 120)]
    ea_raw = df['EA'].astype(float)

    # Bandpass filter at original sampling rate
    ea_filtered = bandpass_filter(ea_raw, 0.1, 5, fs)
    time_ea = df['time[s]'].values

    return time_ea, ea_filtered


# HR Feature Detection

In [17]:
def hr_features_from_window(hr_window):
    hr_window = hr_window.reset_index(drop=True)
    if len(hr_window) < 2:
        # Not enough samples, return zeros
        return {key: 0 for key in [
            'mean_hr', 'median_hr', 'std_hr', 'min_hr', 'max_hr',
            'minRatio_hr', 'maxRatio_hr', 'median_first_derivative',
            'min_first_derivative', 'max_first_derivative',
            'minRatio_first_derivative', 'maxRatio_first_derivative',
            'std_first_derivative', 'min_second_derivative',
            'max_second_derivative', 'std_second_derivative',
            'minRatio_second_derivative', 'maxRatio_second_derivative'
        ]}

    first_diff = hr_window.diff().dropna()
    second_diff = first_diff.diff().dropna()

    def safe_ratio(a, b):
        return a / b if b != 0 else 0

    features = {
        'mean_hr': hr_window.mean(),
        'median_hr': hr_window.median(),
        'std_hr': hr_window.std(),
        'min_hr': hr_window.min(),
        'max_hr': hr_window.max(),
        'minRatio_hr': safe_ratio(hr_window.min(), hr_window.max()),
        'maxRatio_hr': safe_ratio(hr_window.max(), hr_window.min()),
        'median_first_derivative': first_diff.median(),
        'min_first_derivative': first_diff.min(),
        'max_first_derivative': first_diff.max(),
        'minRatio_first_derivative': safe_ratio(first_diff.min(), first_diff.max()),
        'maxRatio_first_derivative': safe_ratio(first_diff.max(), first_diff.min()),
        'std_first_derivative': first_diff.std(),
        'min_second_derivative': second_diff.min(),
        'max_second_derivative': second_diff.max(),
        'std_second_derivative': second_diff.std(),
        'minRatio_second_derivative': safe_ratio(second_diff.min(), second_diff.max()),
        'maxRatio_second_derivative': safe_ratio(second_diff.max(), second_diff.min())
    }
    return features

# Temperature Detection

In [18]:

def temp_features_from_window(temp_window):
    temp_window = temp_window.reset_index(drop=True)

    if len(temp_window) < 2:
        return {key: 0 for key in [
            'mean_temp', 'median_temp', 'std_temp', 'min_temp', 'max_temp',
            'minRatio_temp', 'maxRatio_temp', 'range_temp', 'iqr_temp', 'slope_temp'
        ]}

    def safe_ratio(a, b):
        return a / b if b != 0 else 0

    # deleted the first and second derivative here
    slope = np.polyfit(range(len(temp_window)), temp_window, 1)[0]

    features = {
        'mean_temp': temp_window.mean(),
        'median_temp': temp_window.median(),
        'std_temp': temp_window.std(),
        'min_temp': temp_window.min(),
        'max_temp': temp_window.max(),
        'minRatio_temp': safe_ratio(temp_window.min(), temp_window.max()),
        'maxRatio_temp': safe_ratio(temp_window.max(), temp_window.min()),
        'range_temp': temp_window.max() - temp_window.min(),
        'iqr_temp': temp_window.quantile(0.75) - temp_window.quantile(0.25),
        'slope_temp': slope
    }

    return features

# BI Detection

In [19]:
def bi_features_from_window(bi_window: pd.Series):
    bi_window = bi_window.reset_index(drop=True)
    if len(bi_window) < 2:
        return {key: 0 for key in [
            'mean_bi', 'median_bi', 'std_bi', 'min_bi', 'max_bi',
            'minRatio_bi', 'maxRatio_bi', 'iqr_bi'
        ]}

    first_diff = bi_window.diff().dropna()
    second_diff = first_diff.diff().dropna()

    def safe_ratio(a, b):
        try:
            return float(a) / float(b) if float(b) != 0 else 0.0
        except Exception:
            return 0.0

    features = {
        'mean_bi': bi_window.mean(),
        'median_bi': bi_window.median(),
        'std_bi': bi_window.std(),
        'min_bi': bi_window.min(),
        'max_bi': bi_window.max(),
        'minRatio_bi': safe_ratio(bi_window.min(), bi_window.max()),
        'maxRatio_bi': safe_ratio(bi_window.max(), bi_window.min()),
        'iqr_bi': bi_window.quantile(0.75) - bi_window.quantile(0.25)

    }
    return features

# Windowed Feature Extraction

In [20]:
def windowed_feature_extraction(time_ea, ea_signal, time_hr, hr_signal, time_temp, temp_signal, time_bi, bi_signal, window_sec=5, fs=15, overlap=0.5, label='unknown'):
    window_size = int(window_sec * fs)  # samples per window
    step_size = int(window_size * (1 - overlap))  # step size between windows
    n_samples = len(ea_signal)

    features_list = []
    
    # Create HR dataframe for easy masking
    df_hr = pd.DataFrame({'time[s]': time_hr, 'HR': hr_signal})

    # Create T1 dataframe
    df_temp = pd.DataFrame({'time[s]': time_temp, 'T1': temp_signal})

    # Create BI dataframe
    df_bi = pd.DataFrame({'time[s]': time_bi, 'BI': bi_signal})

    for start in range(0, n_samples - window_size + 1, step_size):
        end = start + window_size

        ea_win = ea_signal[start:end]
        time_win = time_ea[start:end]

        # Process EDA window and extract EDA features
        signals, _ = nk.eda_process(ea_win, sampling_rate=fs, method='neurokit')

        eda_features = {
            'SCR_Onsets_sum': signals['SCR_Onsets'].sum(),
            'SCR_Peaks_sum': signals['SCR_Peaks'].sum(),
            'SCR_Height_mean': signals['SCR_Height'].mean() if len(signals) > 0 else 0,
            'SCR_Amplitude_mean': signals['SCR_Amplitude'].mean() if len(signals) > 0 else 0,
            'SCR_RiseTime_mean': signals['SCR_RiseTime'].mean() if len(signals) > 0 else 0,
            'SCR_Recovery_mean': signals['SCR_Recovery'].mean() if len(signals) > 0 else 0,
            'SCR_RecoveryTime_mean': signals['SCR_RecoveryTime'].mean() if len(signals) > 0 else 0,
        }

        # Get corresponding HR values for same time window (tolerance +/- small epsilon)
        hr_window = df_hr[(df_hr['time[s]'] >= time_win[0]) & (df_hr['time[s]'] <= time_win[-1])]['HR']

        # Skip windows with insufficient HR data
        if len(hr_window) < window_size * 0.8:  # 80% coverage threshold
            continue

        hr_feats = hr_features_from_window(hr_window)

        # Add temperature features
        temp_window = df_temp[(df_temp['time[s]'] >= time_win[0]) & (df_temp['time[s]'] <= time_win[-1])]['T1']
        temp_feats = temp_features_from_window(temp_window)

        # Add BI features
        bi_window = df_bi[(df_bi['time[s]'] >= time_win[0]) & (df_bi['time[s]'] <= time_win[-1])]['BI']
        bi_feats = bi_features_from_window(bi_window)

        combined_features = {**eda_features, **hr_feats, **temp_feats, **bi_feats, 'label': label}

        features_list.append(combined_features)

    return pd.DataFrame(features_list)

# ML Model

In [ ]:
def pred_tree(frames, window_sec):
    X = frames.drop('label', axis=1) #drops 'label' column
    y = frames['label']

    #splits into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify = y, random_state=42)

    # dt_model = DecisionTreeClassifier(criterion='entropy', max_depth=3)

    # #trains model
    # dt_model.fit(X_train, y_train)

    # #predicts 'y' values with test 'x' values
    # y_pred = dt_model.predict(X_test)

    # #checks accuracy of predicted 'y' against true 'y'
    # acc = accuracy_score(y_test, y_pred)

    # # precision, recall, f1 score
    # print(classification_report(y_test, y_pred))

    # print("Decision Tree Accuracy:", acc)

    # print("--------------------------------------------")

    rf_model = RandomForestClassifier(n_estimators=1, random_state=42, class_weight = "balanced")
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_test)

    # print(classification_report(y_test, rf_pred))

    # confusion matrix
    # rf_cm = confusion_matrix(y_test, rf_pred, labels=rf_model.classes_)
    
    # disp = ConfusionMatrixDisplay(confusion_matrix = rf_cm, display_labels = rf_model.classes_)
    # disp.plot()

    print(f" {window_sec} secs Random Forest Accuracy:", accuracy_score(y_test, rf_pred))

    # cross validation below, only run this code when RandomForestClassifier n_estimators is set to 1

    # loo = LeaveOneOut()
    # scores = cross_val_score(rf_model, X, y, scoring='accuracy', cv=loo, n_jobs=-1)
    # print('Accuracy: %.3f (%.3f)' % ((scores).mean(), (scores).std()))

# Code to Print Graphs:

In [22]:
# def graph(raw_data):
#     df_min = raw_data["LocalTimestamp"].min() + 200
#     df_max = raw_data["LocalTimestamp"].max() - 200
#     df = raw_data[(raw_data["LocalTimestamp"]>df_min) & (raw_data["LocalTimestamp"]<df_max)]

#     df_ea = df["EA"]

#     nk.signal_plot(df_ea, sampling_rate=15) # Check basic graph

# Running Program . . . 

In [23]:
# Run detection

fs_eda = 15
# window_sec = 55
overlap = 0.5

for window_sec in range(5, 61, 5):
    data = pd.DataFrame()

    # Engaged files
    engaged_filenames = glob.glob("engaged_EA/*.csv")
    for file in engaged_filenames:
        time_ea, ea_filtered = ea_detection(file, fs=fs_eda)

        x = file.split("_")
        y = x[1].split("\\")
        hr_name = "engaged_HR" + "\\" + y[1] + "_" + x[2] + "_HR.csv"

        df_hr = pd.read_csv(hr_name)
        df_hr['time[s]'] = (df_hr['LocalTimestamp'] - df_hr['LocalTimestamp'].iloc[0])
        df_hr = df_hr.loc[(df_hr['time[s]'] > 120) & (df_hr['time[s]'] < (df_hr['time[s]'].iloc[-1]) - 120)]

        # Upsample HR to 15 Hz to align with EDA
        time_hr_up, hr_up = upsample_signal(df_hr['time[s]'].values, df_hr['HR'].values, fs_new=fs_eda)

        # New add for integrate T1 features
        temp_name = "engaged_T1" + "\\" + y[1] + "_" + x[2] + "_T1.csv" 
        df_temp   = pd.read_csv(temp_name)
        df_temp['time[s]'] = (df_temp['LocalTimestamp'] - df_temp['LocalTimestamp'].iloc[0])
        df_temp = df_temp.loc[(df_temp['time[s]'] > 120) & (df_temp['time[s]'] < (df_temp['time[s]'].iloc[-1]) - 120)]

        time_temp_up, temp_up = upsample_signal(df_temp['time[s]'].values, df_temp['T1'].values, fs_new=fs_eda)

        #BI dataframe
        bi_name = "engaged_BI" + "/" + y[1] + "_" + x[2] + "_BI.csv" 
        df_bi   = pd.read_csv(bi_name)
        df_bi['time[s]'] = (df_bi['LocalTimestamp'] - df_bi['LocalTimestamp'].iloc[0])
        df_bi = df_bi.loc[(df_bi['time[s]'] > 120) & (df_bi['time[s]'] < (df_bi['time[s]'].iloc[-1]) - 120)]
        time_bi_up, bi_up = upsample_signal(df_bi['time[s]'].values, df_bi['BI'].values, fs_new=fs_eda)

        windowed_df = windowed_feature_extraction(time_ea, ea_filtered, time_hr_up, hr_up, time_temp_up, temp_up, time_bi_up, bi_up,
                                                    window_sec=window_sec, fs=fs_eda,
                                                    overlap=overlap, label='engaged')
        
        # print(f"engaged file {file} generated {len(windowed_df)} windowed samples")

        if not windowed_df.empty:
            data = pd.concat([data, windowed_df], ignore_index=True)
        else:
            print(f"Warning: No valid windows extracted from engaged file {file}")

        data = pd.concat([data, windowed_df], ignore_index=True)
        
    # print(len(data))

    # Relaxed files
    relaxed_filenames = glob.glob("relaxed_EA/*.csv")
    for file in relaxed_filenames:
        time_ea, ea_filtered = ea_detection(file, fs=fs_eda)

        x = file.split("_")
        y = x[1].split("\\")
        hr_name = "relaxed_HR" + "\\" + y[1] + "_" + x[2] + "_HR.csv"

        df_hr = pd.read_csv(hr_name)
        df_hr['time[s]'] = (df_hr['LocalTimestamp'] - df_hr['LocalTimestamp'].iloc[0])

        # Upsample HR to 15 Hz to align with EDA
        time_hr_up, hr_up = upsample_signal(df_hr['time[s]'].values, df_hr['HR'].values, fs_new=fs_eda)

        temp_name = "relaxed_T1" + "\\" + y[1] + "_" + x[2] + "_T1.csv"
        df_temp   = pd.read_csv(temp_name)
        df_temp['time[s]'] = (df_temp['LocalTimestamp'] - df_temp['LocalTimestamp'].iloc[0])
        time_temp_up, temp_up = upsample_signal(df_temp['time[s]'].values, df_temp['T1'].values, fs_new=fs_eda)

        #BI dataframe
        bi_name = "relaxed_BI" + "/" + y[1] + "_" + x[2] + "_BI.csv" 
        df_bi   = pd.read_csv(bi_name)
        df_bi['time[s]'] = (df_bi['LocalTimestamp'] - df_bi['LocalTimestamp'].iloc[0])
        df_bi = df_bi.loc[(df_bi['time[s]'] > 120) & (df_bi['time[s]'] < (df_bi['time[s]'].iloc[-1]) - 120)]
        time_bi_up, bi_up = upsample_signal(df_bi['time[s]'].values, df_bi['BI'].values, fs_new=fs_eda)

        # Extract features with windowing
        windowed_df = windowed_feature_extraction(time_ea, ea_filtered, time_hr_up, hr_up, time_temp_up, temp_up, time_bi_up, bi_up,
                                                    window_sec=window_sec, fs=fs_eda,
                                                    overlap=overlap, label='relaxed')
        
        # print(f"engaged file {file} generated {len(windowed_df)} windowed samples")

        if not windowed_df.empty:
            data = pd.concat([data, windowed_df], ignore_index=True)
        else:
            print(f"Warning: No valid windows extracted from engaged file {file}")

    # print(f"Total samples: {len(data)}")
    pred_tree(data, window_sec)
# data.head(1)

c:\Users\omshr\anaconda3\envs\cs328\lib\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])
c:\Users\omshr\anaconda3\envs\cs328\lib\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])
c:\Users\omshr\anaconda3\envs\cs328\lib\site-packages\neurokit2\eda\eda_peaks.py:127: RuntimeWarning: All-NaN slice encountered
  info["SCR_Peaks"] > np.nanmin(info["SCR_Onsets"]), ~np.isnan(info["SCR_Onsets"])


              precision    recall  f1-score   support

     engaged       0.87      0.91      0.89       622
     relaxed       0.81      0.75      0.78       323

    accuracy                           0.85       945
   macro avg       0.84      0.83      0.83       945
weighted avg       0.85      0.85      0.85       945

Decision Tree Accuracy: 0.852910052910053
--------------------------------------------
 5 secs Random Forest Accuracy: 0.8698412698412699


KeyboardInterrupt: 

# CHECK SAMPLING RATE

In [ ]:
# #just edit file path to check sampling rate
# file_path ="engaged_HR\\2025-07-15_19-32-05-631421_HR.csv"

# df = pd.read_csv(file_path)
# df['time[s]'] = (df['LocalTimestamp'] - df['LocalTimestamp'].iloc[0])
# time = df['time[s]'].max() - df['time[s]'].min()

# print(f"SAMPLING RATE: {len(df)/time}")